In [1]:
import requests
import json
import os
from dotenv import load_dotenv

# 1. Carrega as variáveis escondidas no arquivo .env
load_dotenv()

# 2. Pega a chave de forma segura
minha_chave = os.getenv("API_KEY_FOOTBALL")

# 3. Definindo a URL e Parâmetros
url = "https://v3.football.api-sports.io/fixtures"
parametros = {
    "league": "71",
    "season": "2023"
}

# 4. O seu crachá de acesso usando a variável segura
headers = {
    "x-apisports-key": minha_chave
}

# 5. Fazendo a requisição
resposta = requests.get(url, headers=headers, params=parametros)
dados = resposta.json()

# 6. Validando o que chegou
if 'results' in dados and dados['results'] > 0:
    print(f"Sucesso! Total de partidas retornadas pela API: {dados['results']}")
    primeira_partida = dados['response'][0]
    print(json.dumps(primeira_partida, indent=2))
else:
    print("Nenhum dado retornado ou erro na conexão. Verifique o retorno da API:")
    print(dados)

Sucesso! Total de partidas retornadas pela API: 380
{
  "fixture": {
    "id": 1005651,
    "referee": "Paulo Cesar Zanovelli da Silva",
    "timezone": "UTC",
    "date": "2023-04-15T19:00:00+00:00",
    "timestamp": 1681585200,
    "periods": {
      "first": 1681585200,
      "second": 1681588800
    },
    "venue": {
      "id": 258,
      "name": "Allianz Parque",
      "city": "S\u00e3o Paulo, S\u00e3o Paulo"
    },
    "status": {
      "long": "Match Finished",
      "short": "FT",
      "elapsed": 90,
      "extra": null
    }
  },
  "league": {
    "id": 71,
    "name": "Serie A",
    "country": "Brazil",
    "logo": "https://media.api-sports.io/football/leagues/71.png",
    "flag": "https://media.api-sports.io/flags/br.svg",
    "season": 2023,
    "round": "Regular Season - 1",
    "standings": true
  },
  "teams": {
    "home": {
      "id": 121,
      "name": "Palmeiras",
      "logo": "https://media.api-sports.io/football/teams/121.png",
      "winner": true
    },
    "

In [5]:
import pandas as pd

# 1. Isolando apenas a lista de partidas (que está dentro da chave 'response')
lista_times = dados['response']

# 2. Criando uma lista vazia que vai guardar nossas linhas tratadas
dados_estruturados = []

# 3. O "Trator": Passando partida por partida e extraindo só o que importa
for partida in lista_times:
    linha = {
        "id_partida": partida['fixture']['id'],
        "data_jogo": partida['fixture']['date'],
        "id_time_mandante": partida['teams']['home']['id'],
        "id_time_visitante": partida['teams']['away']['id'],
        "gols_mandante": partida['goals']['home'],
        "gols_visitante": partida['goals']['away'],
        "id_estadio": partida['fixture']['venue']['id']
    }
    dados_estruturados.append(linha)

# 4. A Mágica do Pandas: Transformando a lista em um DataFrame (Tabela)
df_partidas = pd.DataFrame(dados_estruturados)

# 5. Exibindo as 5 primeiras linhas da nossa nova tabela limpa
print("Transformação concluída com sucesso! Veja a tabela:")
display(df_partidas.head()) # No Jupyter, o 'display' renderiza uma tabela visualmente mais bonita que o 'print'

Transformação concluída com sucesso! Veja a tabela:


,id_partida,data_jogo,id_time_mandante,id_time_visitante,gols_mandante,gols_visitante,id_estadio
0,1005651,2023-04-15T19:00:00+00:00,121,1193,2,1,258.0
1,1005658,2023-04-15T19:00:00+00:00,125,124,0,3,206.0
2,1005650,2023-04-15T21:30:00+00:00,120,126,2,1,218.0
3,1005656,2023-04-15T21:30:00+00:00,134,151,2,0,10493.0
4,1005657,2023-04-15T21:30:00+00:00,154,119,1,1,225.0


In [6]:
import sqlite3

# 1. Estabelecendo a conexão (o Python cria o arquivo do banco automaticamente na sua pasta)
conexao = sqlite3.connect('banco_brasileirao.db')

# 2. A Carga: Enviando o DataFrame do Pandas direto para a tabela do banco
# O parâmetro if_exists='replace' garante que, se rodarmos amanhã, ele atualiza a tabela sem duplicar tudo
df_partidas.to_sql(name='fato_partidas', con=conexao, if_exists='replace', index=False)

# 3. Fechando a conexão por segurança
conexao.close()

print("Carga concluída com sucesso! Os dados estão blindados no banco local.")

Carga concluída com sucesso! Os dados estão blindados no banco local.
